In [9]:
import platform

# Install openvino-dev package
%pip install -q "openvino>=2023.1.0" opencv-python tqdm

if platform.system() != "Windows":
    %pip install -q "matplotlib>=3.4"
else:
    %pip install -q "matplotlib>=3.4,<3.7"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
from collections import namedtuple
from itertools import groupby

import cv2
import matplotlib.pyplot as plt
import numpy as np
import openvino as ov

# Fetch `notebook_utils` module
import requests

r = requests.get(
    url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
)

open("notebook_utils.py", "w").write(r.text)
from notebook_utils import download_file, device_widget

In [18]:
# Directories where data will be placed.
base_models_dir = "models"
data_folder = "data"
charlist_folder = f"{data_folder}/text"

# Precision used by the model.
precision = "FP16"

In [19]:
# Check if the open_model_zoo directory exists
!ls


3D-pose-estimation-webcam
3D-segmentation-point-clouds
action-recognition-webcam
amused-lightweight-text-to-image
animate-anyone
async-api
auto-device
bark-text-to-audio
big-transfer-quantization
blip-visual-language-processing
clip-language-saliency-map
clip-zero-shot-image-classification
colour_detection.ipynb
controlnet-stable-diffusion
convert-to-openvino
cross-lingual-books-alignment
ct-segmentation-quantize
ddcolor-image-colorization
depth-anything
detectron2-to-openvino
distilbert-sequence-classification
distil-whisper-asr
dolly-2-instruction-following
dynamicrafter-animating-images
efficient-sam
encodec-audio-compression
explainable-ai-1-basic
explainable-ai-2-deep-dive
explainable-ai-3-map-interpretation
Face_REG
fast-segment-anything
film-slowmo
florence2
flux.1-image-generation
freevc-voice-conversion
gpu-device
grammar-correction
grounded-segment-anything
handwritten-ocr
hello-detection
hello-npu
hello-segmentation
hello-world
hugging-face-hub
hunyuan-dit-image-generation
i

In [20]:
import cv2
import numpy as np
from openvino.runtime import Core

# Initialize OpenVINO Runtime
ie = Core()

# Load models
face_detection_model = ie.read_model("models/face-detection-adas-0001.xml")
face_recognition_model = ie.read_model("models/face-reidentification-retail-0095.xml")

# Compile models for CPU
compiled_face_detection_model = ie.compile_model(face_detection_model, "CPU")
compiled_face_recognition_model = ie.compile_model(face_recognition_model, "CPU")

# Load input image
image = cv2.imread('input_image.jpg')

# Prepare the image for the network
input_image = cv2.resize(image, (672, 384)) # Resize image to model's input size
input_image = input_image.transpose((2, 0, 1)) # Change data layout from HWC to CHW
input_image = input_image[np.newaxis, :] # Add batch dimension

# Run face detection
face_detection_output = compiled_face_detection_model([input_image])[compiled_face_detection_model.outputs[0]]

# Process the output
for detection in face_detection_output[0][0]:
    if detection[2] > 0.5:  # Confidence threshold
        xmin, ymin, xmax, ymax = (detection[3:7] * np.array([image.shape[1], image.shape[0], image.shape[1], image.shape[0]])).astype(int)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)

cv2.imshow("Face Detection", image)
cv2.waitKey(0)


RuntimeError: Exception from src/inference/src/cpp/core.cpp:90:
Check 'util::directory_exists(path) || util::file_exists(path)' failed at src/frontends/common/src/frontend.cpp:113:
FrontEnd API failed with GeneralFailure:
ir: Could not open the file: "models/face-detection-adas-0001.xml"

